# Chapter 1 — Common Conventions and API Elements of scikit-learn

**Reference:** John Sukup, *scikit-learn Cookbook*, Third Edition, Chapter 1.

## Objectives
This chapter introduces scikit-learn's consistent API: estimators, transformers, custom components, pipelines, common attributes and methods, hyperparameter search, estimator tags, and API best practices.

The examples use synthetic data and scikit-learn's built-in Iris dataset, so no external files are required. Run the notebook from top to bottom in Google Colab.

## 1. Environment and imports

The book targets Python 3.9+ and scikit-learn 1.5+. This cell checks the versions available in the current runtime.

In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

## 2. scikit-learn's design philosophy

scikit-learn emphasizes consistency, simplicity, modularity, and reusability. The common methods are:

- `fit(X, y)`: learn from training data.
- `predict(X)`: predict targets for input features.
- `transform(X)`: apply a learned transformation to features.
- `fit_transform(X)`: fit a transformer and transform data in one operation.

`X` is normally a feature matrix (rows are observations, columns are features); `y` contains target values in supervised learning.

In [ ]:
from sklearn.linear_model import LinearRegression

X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1.0, 2.0, 2.8, 4.2, 5.1])

model = LinearRegression()
model.fit(X, y)

X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
display(pd.DataFrame({"X": X_new.ravel(), "Predicted y": predictions}))

**Interpretation:** `fit()` estimates a relationship from training examples. `predict()` applies that learned relationship to new inputs; predictions are estimates, not guaranteed true outcomes.

## 3. Understanding estimators

An **estimator** is an object that learns information from data. A fitted estimator often stores learned attributes ending in an underscore, such as `coef_` and `intercept_`. Some unsupervised estimators also provide `fit_predict()`, which fits the model and returns assignments such as cluster labels.

In [ ]:
print("Coefficient(s):", model.coef_)
print("Intercept:", round(model.intercept_, 3))
print("Training R-squared:", round(model.score(X, y), 3))

**Interpretation:** A linear coefficient describes the change in predicted target for a one-unit increase in a feature. For regression, `score()` normally returns R², the coefficient of determination.

## 4. Transformers and `transform()`

A **transformer** learns a data-processing rule and applies it to features. `StandardScaler`, for example, learns feature means and standard deviations. Fit preprocessing on training data only, then apply the fitted transformer to validation/test data to avoid information leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scale = np.array([[10.0, 100.0], [20.0, 200.0], [30.0, 300.0]])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_scale)

print("Original features:")
display(pd.DataFrame(X_scale, columns=["Feature A", "Feature B"]))
print("Standardized features:")
display(pd.DataFrame(X_scaled, columns=["Feature A", "Feature B"]).round(3))
print("Learned means:", scaler.mean_)

**Interpretation:** Standardization centers each feature around zero and scales it based on training data. Scaling is particularly important for algorithms sensitive to feature magnitude, including nearest-neighbor methods and many regularized linear models.

## 5. Custom estimators and transformers

Custom components can follow the same API as built-in scikit-learn components. A transformer typically implements `fit()` and `transform()`, and `fit()` returns `self`. Inheriting from `BaseEstimator` and `TransformerMixin` helps provide compatibility with pipelines and model-selection utilities.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class AddConstantTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, constant=1.0):
        self.constant = constant

    def fit(self, X, y=None):
        X = np.asarray(X)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self, "n_features_in_")
        X = np.asarray(X)
        if X.shape[1] != self.n_features_in_:
            raise ValueError("Input feature count differs from fit data.")
        return X + self.constant

X_custom = np.array([[1, 2], [3, 4]])
transformer = AddConstantTransformer(constant=5)
X_result = transformer.fit_transform(X_custom)
display(pd.DataFrame(X_result, columns=["A after transformation", "B after transformation"]))

**Interpretation:** The transformer adds its configured constant to every feature. The example is intentionally simple; it demonstrates the expected interface and how learned attributes conventionally end with `_`.

## 6. Pipelines and workflow automation

A **pipeline** chains transformers and a final estimator into one object. Fitting proceeds through each preprocessing step and then the model. Prediction applies the fitted transformations before generating predictions. Pipelines also help prevent leakage when used with cross-validation because preprocessing is fitted within each training fold.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

iris = load_iris(as_frame=True)
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("Pipeline steps:", list(pipeline.named_steps.keys()))

**Interpretation:** The scaler and classifier form one workflow. The test set is held out from fitting, so its score estimates performance on observations not used to train the model.

## 7. Common attributes and methods

Estimators often provide `get_params()` and `set_params()` for configuration, plus `score()` for a default evaluation metric. Learned attributes vary by estimator. Model-selection utilities use parameter names to configure components, including nested pipeline steps.

In [ ]:
params = pipeline.get_params()
for name in ["scaler__with_mean", "classifier__C", "classifier__max_iter"]:
    print(f"{name}: {params[name]}")

print("Fitted classes:", pipeline.named_steps["classifier"].classes_)
print(f"Pipeline test score: {pipeline.score(X_test, y_test):.3f}")

## 8. Hyperparameter tuning with search methods

**Parameters** are learned during fitting; **hyperparameters** are configuration choices, such as logistic regression's `C`. `GridSearchCV` evaluates the combinations in a parameter grid using cross-validation and selects the best mean validation score. Tune using training data and reserve the test set for final evaluation.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__C": [0.1, 1.0, 10.0],
    "classifier__solver": ["lbfgs"]
}
grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best mean CV accuracy: {grid_search.best_score_:.3f}")
print(f"Test accuracy after tuning: {grid_search.score(X_test, y_test):.3f}")

**Interpretation:** Cross-validation compares configurations over multiple splits. The held-out test score evaluates the selected model. Tuning does not guarantee an improvement on every dataset.

## 9. Estimator metadata and tags

Estimator **tags** describe capabilities and behavior, helping scikit-learn's testing and workflow tools interact with estimators. The API has changed across versions, so this version-aware example inspects tags if the installed release exposes the public `get_tags` utility.

In [ ]:
tag_model = LogisticRegression()
try:
    from sklearn.utils import get_tags
    tags = get_tags(tag_model)
    print("Estimator tags are available.")
    print("Tag object type:", type(tags).__name__)
    print(tags)
except (ImportError, AttributeError, TypeError):
    print("Public get_tags utility is unavailable in this runtime.")
    print("scikit-learn version:", sklearn.__version__)

**Interpretation:** Tags are metadata, not predictions or performance scores. Their exact interface depends on the installed scikit-learn version.

## 10. Best practices for API usage

1. Use the standard `fit()`, `predict()`, and `transform()` interfaces where appropriate.
2. Combine preprocessing and modeling with a pipeline.
3. Split data before fitting preprocessing or models.
4. Use cross-validation for model selection; reserve the test set for final evaluation.
5. Set random seeds where supported for reproducibility.
6. Inspect learned attributes and evaluation metrics rather than assuming a model performs well.
7. Validate inputs and follow scikit-learn conventions when creating custom components.
8. Check documentation when APIs change between versions.

## 11. Chapter summary

This chapter introduced the conventions that make scikit-learn components reusable. Estimators learn from data, transformers modify features, and pipelines combine preprocessing with modeling. Common methods and attributes make estimators easier to inspect; `GridSearchCV` supports systematic hyperparameter selection; and tags provide metadata for interoperability.

| Term | Meaning |
|---|---|
| Estimator | Object that learns from data through `fit()` |
| Transformer | Component that learns and applies a feature transformation |
| Pipeline | Ordered chain of transformers and a final estimator |
| Parameter | Value learned during model fitting |
| Hyperparameter | Configuration selected before fitting or through model search |
| Cross-validation | Repeated training/validation splits used to estimate performance |
| Estimator tags | Metadata describing capabilities and behavior |

## Reference

Sukup, J. (2025). *scikit-learn Cookbook* (3rd ed.). Packt Publishing. Chapter 1, “Common Conventions and API Elements of scikit-learn.” ISBN 978-1-83664-445-3.

Official companion code repository: https://github.com/PacktPublishing/scikit-learn-Cookbook-Third-Edition